In [1]:
import os

ucl_path = r"C:\Users\Asus\OneDrive\Desktop\neopain-research\data\raw\UCL"

for root, dirs, files in os.walk(ucl_path):
    level = root.replace(ucl_path, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    sub_indent = '  ' * (level + 1)
    for f in sorted(files)[:10]:  # first 10 files per folder
        size_kb = os.path.getsize(os.path.join(root, f)) // 1024
        print(f"{sub_indent}📄 {f}  ({size_kb} KB)")
    if len(files) > 10:
        print(f"{sub_indent}... and {len(files)-10} more files")

📁 UCL/
  📁 Database/
    📄 (1) Infant Demographics.xlsx  (68 KB)
    📄 (2) Study details.xlsx  (31 KB)
    📄 (3) Stimulation information.xlsx  (65 KB)
    📄 (4) Infant patient notes.xlsx  (189 KB)
    📄 (5) Maternal patient notes.xlsx  (18 KB)
  📁 EEG/
    📁 2510001/
      📄 2510001A01.mat  (596 KB)
      📄 2510001C01.mat  (596 KB)
      📄 2510001L01.mat  (7374 KB)
    📁 2510101/
      📄 2510101A01.mat  (625 KB)
      📄 2510101C01.mat  (625 KB)
      📄 2510101L01.mat  (7818 KB)
    📁 2510201/
      📄 2510201A01.mat  (605 KB)
      📄 2510201C01.mat  (609 KB)
      📄 2510201L01.mat  (7765 KB)
    📁 2510301/
      📄 2510301A01.mat  (625 KB)
      📄 2510301C01.mat  (626 KB)
      📄 2510301L01.mat  (7202 KB)
    📁 2510401/
      📄 2510401A01.mat  (627 KB)
      📄 2510401C01.mat  (631 KB)
      📄 2510401L01.mat  (7717 KB)
    📁 2510501/
      📄 2510501A01.mat  (546 KB)
      📄 2510501C01.mat  (550 KB)
      📄 2510501L01.mat  (6750 KB)
    📁 2510601/
      📄 2510601A01.mat  (643 KB)
      📄 2

In [2]:
import scipy.io
import numpy as np

ucl_eeg_path = r"C:\Users\Asus\OneDrive\Desktop\neopain-research\data\raw\UCL\EEG"

# Load one heel lance file
mat = scipy.io.loadmat(f"{ucl_eeg_path}\\2510001\\2510001L01.mat")

# See all variables inside
print("Keys in .mat file:")
for key, val in mat.items():
    if not key.startswith('_'):
        if hasattr(val, 'shape'):
            print(f"  {key}: shape={val.shape}, dtype={val.dtype}")
        else:
            print(f"  {key}: {type(val)} = {val}")

Keys in .mat file:
  EEG: shape=(1, 1), dtype=[('setname', 'O'), ('filename', 'O'), ('filepath', 'O'), ('subject', 'O'), ('condition', 'O'), ('session', 'O'), ('nbchan', 'O'), ('trials', 'O'), ('pnts', 'O'), ('srate', 'O'), ('xmin', 'O'), ('xmax', 'O'), ('times', 'O'), ('data', 'O'), ('icaact', 'O'), ('icawinv', 'O'), ('icasphere', 'O'), ('icaweights', 'O'), ('icachansind', 'O'), ('chanlocs', 'O'), ('urchanlocs', 'O'), ('chaninfo', 'O'), ('ref', 'O'), ('event', 'O'), ('urevent', 'O'), ('eventdescription', 'O'), ('epoch', 'O'), ('epochdescription', 'O'), ('reject', 'O'), ('stats', 'O'), ('specdata', 'O'), ('specicaact', 'O'), ('splinefile', 'O'), ('icasplinefile', 'O'), ('dipfit', 'O'), ('history', 'O'), ('saved', 'O'), ('etc', 'O'), ('other_data', 'O')]


In [3]:
import scipy.io
import numpy as np
import pandas as pd
import os

ucl_eeg_path = r"C:\Users\Asus\OneDrive\Desktop\neopain-research\data\raw\UCL\EEG"
db_path      = r"C:\Users\Asus\OneDrive\Desktop\neopain-research\data\raw\UCL\Database"

# ── STEP 1: Correctly load one .mat file ─────────────────────────────
mat = scipy.io.loadmat(
    f"{ucl_eeg_path}\\2510001\\2510001L01.mat",
    squeeze_me=True,
    struct_as_record=False
)
eeg = mat['EEG']

print("=" * 50)
print("SINGLE .MAT FILE INSPECTION")
print("=" * 50)
print(f"Subject       : {eeg.subject}")
print(f"Condition     : {eeg.condition}")
print(f"Sampling rate : {eeg.srate} Hz")
print(f"Channels      : {eeg.nbchan}")
print(f"Time points   : {eeg.pnts}")
print(f"Duration      : {eeg.pnts / eeg.srate:.1f} seconds")
print(f"EEG data shape: {eeg.data.shape}")
print(f"Time axis     : {eeg.times[0]:.3f}s to {eeg.times[-1]:.3f}s")

# Channel names
try:
    ch_names = [str(eeg.chanlocs[i].labels) for i in range(len(eeg.chanlocs))]
    print(f"Channel names : {ch_names}")
except Exception as ex:
    print(f"Channel names : could not parse — {ex}")

# other_data — HR and SpO2 pre-computed values
print(f"\nother_data type  : {type(eeg.other_data)}")
try:
    od = eeg.other_data
    if hasattr(od, 'dtype') and od.dtype.names:
        print(f"other_data fields: {od.dtype.names}")
        for name in od.dtype.names:
            print(f"  {name}: {od[name]}")
    else:
        print(f"other_data shape : {od.shape}")
        print(f"other_data value : {od}")
except Exception as ex:
    print(f"other_data error : {ex}")


SINGLE .MAT FILE INSPECTION
Subject       : 25100
Condition     : lance
Sampling rate : 2000 Hz
Channels      : 18
Time points   : 8000
Duration      : 4.0 seconds
EEG data shape: (18, 8000)
Time axis     : -2000.000s to 1999.500s
Channel names : ['Cz', 'CPz', 'F8', 'T8', 'TP10', 'P8', 'O2', 'F4', 'C4', 'CP4', 'F7', 'T7', 'TP9', 'P7', 'O1', 'F3', 'C3', 'CP3']

other_data type  : <class 'scipy.io.matlab._mio5_params.mat_struct'>
other_data error : 'mat_struct' object has no attribute 'shape'


In [4]:
import pandas as pd

db_path = r"C:\Users\Asus\OneDrive\Desktop\neopain-research\data\raw\UCL\Database"

# File 1 — all sheets
xl1 = pd.ExcelFile(f"{db_path}\\(1) Infant Demographics.xlsx")
print("File 1 sheets:", xl1.sheet_names)

# File 2 — all sheets
xl2 = pd.ExcelFile(f"{db_path}\\(2) Study details.xlsx")
print("File 2 sheets:", xl2.sheet_names)

# File 3 — all sheets (3 tabs)
xl3 = pd.ExcelFile(f"{db_path}\\(3) Stimulation information.xlsx")
print("File 3 sheets:", xl3.sheet_names)

# File 4 — all sheets
xl4 = pd.ExcelFile(f"{db_path}\\(4) Infant patient notes.xlsx")
print("File 4 sheets:", xl4.sheet_names)

# File 5 — all sheets
xl5 = pd.ExcelFile(f"{db_path}\\(5) Maternal patient notes.xlsx")
print("File 5 sheets:", xl5.sheet_names)

File 1 sheets: ['Demographics', 'Delivery details', 'SNAP scores']
File 2 sheets: ['Study context', 'EEG details']
File 3 sheets: ['Heel lance', 'Sham control', 'Auditory control']
File 4 sheets: ['Ventilation', 'Diagnosis', 'Cranial scans', 'Medication', 'Heel lances', 'Painful procedures', 'Injuries']
File 5 sheets: ['Maternal']


In [3]:
import scipy.io
import numpy as np
import pandas as pd
import os

ucl_eeg_path = r"C:\Users\Asus\OneDrive\Desktop\neopain-research\data\raw\UCL\EEG"
db_path      = r"C:\Users\Asus\OneDrive\Desktop\neopain-research\data\raw\UCL\Database"
save_path    = r"C:\Users\Asus\OneDrive\Desktop\neopain-research\data\processed"

# ── Part A: Fix other_data access ────────────────────────────────────
mat = scipy.io.loadmat(
    f"{ucl_eeg_path}\\2510001\\2510001L01.mat",
    squeeze_me=True, struct_as_record=False
)
eeg = mat['EEG']
od  = eeg.other_data

print("=== other_data fields ===")
print(od._fieldnames)
for field in od._fieldnames:
    val = getattr(od, field)
    print(f"  {field}: {val}")

# ── Part B: Load all 3 PIPP sheets with correct headers ──────────────
print("\n=== LOADING ALL 3 PIPP SHEETS ===")

lance_df    = pd.read_excel(f"{db_path}\\(3) Stimulation information.xlsx",
                             sheet_name='Heel lance',      header=1)
sham_df     = pd.read_excel(f"{db_path}\\(3) Stimulation information.xlsx",
                             sheet_name='Sham control',    header=1)
auditory_df = pd.read_excel(f"{db_path}\\(3) Stimulation information.xlsx",
                             sheet_name='Auditory control', header=1)

print(f"Heel lance rows   : {len(lance_df)}")
print(f"Sham control rows : {len(sham_df)}")
print(f"Auditory rows     : {len(auditory_df)}")

pipp_col = 'Total PIPP score'
print(f"\nHeel lance — PIPP available: "
      f"{lance_df[pipp_col].notna().sum()} / {len(lance_df)}")
print(f"  PIPP range : {lance_df[pipp_col].min():.0f} – "
      f"{lance_df[pipp_col].max():.0f}")
print(f"  PIPP mean  : {lance_df[pipp_col].mean():.1f}")
print(f"  Pain (≥7)  : {(lance_df[pipp_col] >= 7).sum()}")
print(f"  No pain(<7): {(lance_df[pipp_col] < 7).sum()}")

# ── Part C: Load medication sheet to flag analgesic use ──────────────
print("\n=== MEDICATION CHECK ===")
meds_df = pd.read_excel(f"{db_path}\\(4) Infant patient notes.xlsx",
                         sheet_name='Medication', header=0)
print(f"Medication sheet columns: {list(meds_df.columns[:8])}")
print(meds_df.head(3))

# ── Part D: Load demographics for gestational age ────────────────────
print("\n=== DEMOGRAPHICS ===")
demo_df = pd.read_excel(f"{db_path}\\(1) Infant Demographics.xlsx",
                         sheet_name='Demographics', header=0)
print(f"Columns: {list(demo_df.columns)}")
ga_col = 'Gestational age at birth (weeks)'
print(f"GA range: {demo_df[ga_col].min():.1f} – "
      f"{demo_df[ga_col].max():.1f} weeks")

# ── Part E: Build master label table ─────────────────────────────────
print("\n=== BUILDING MASTER LABEL TABLE ===")

lance_df['condition']    = 'lance';    lance_df['pain_label'] = 1
sham_df['condition']     = 'sham';     sham_df['pain_label']  = 0
auditory_df['condition'] = 'auditory'; auditory_df['pain_label'] = 0

master = pd.concat([lance_df, sham_df, auditory_df], ignore_index=True)

# Refined PIPP-based label where available
master['pipp_label'] = master[pipp_col].apply(
    lambda x: 1 if pd.notna(x) and x >= 7
              else (0 if pd.notna(x) else np.nan)
)

# Build EEG file path for each row
def build_mat_path(row):
    fname = str(row['EEG file name']).strip()
    subj  = str(int(row['Record identifier']))
    # Find the folder — subject folders have varying digit lengths
    return f"{ucl_eeg_path}\\{subj}\\{fname}.mat"

master['mat_path'] = master.apply(build_mat_path, axis=1)
master['file_exists'] = master['mat_path'].apply(os.path.exists)

keep_cols = ['Record identifier', 'EEG file name', 'condition',
             'pain_label', pipp_col, 'pipp_label',
             'Baseline heart rate (bmp)', 'Max heart rate (bpm)',
             'Baseline O2 sats', 'Min O2 sats',
             'mat_path', 'file_exists']
keep_cols = [c for c in keep_cols if c in master.columns]
master_clean = master[keep_cols].copy()

print(f"Total records       : {len(master_clean)}")
print(f"Files found on disk : {master_clean['file_exists'].sum()}")
print(f"Files NOT found     : {(~master_clean['file_exists']).sum()}")
print(f"\nCondition breakdown:")
print(master_clean['condition'].value_counts())
print(f"\nPain label breakdown:")
print(master_clean['pain_label'].value_counts())
print(f"\nSample rows (heel lance):")
print(master_clean[master_clean['condition']=='lance'].head(99).to_string())

# Save
master_clean.to_csv(f"{save_path}\\ucl_master_labels.csv", index=False)
print(f"\n✅ Saved ucl_master_labels.csv — {len(master_clean)} rows")

=== other_data fields ===
['data', 'labels']
  data: [[ 2.1012848e+05  2.1013442e+05  2.1020109e+05 ...  2.0997173e+05
   2.0997011e+05  2.0997300e+05]
 [ 2.8656946e+03  2.8636472e+03  2.8639285e+03 ...  2.5890908e+03
   2.5874958e+03  2.5822041e+03]
 [ 8.7915604e+01  8.8012260e+01  8.6382843e+01 ... -5.6784859e+01
  -5.7307503e+01 -5.9326843e+01]]
  labels: ['RECG' 'LECG' 'Resp']

=== LOADING ALL 3 PIPP SHEETS ===
Heel lance rows   : 112
Sham control rows : 99
Auditory rows     : 99

Heel lance — PIPP available: 79 / 112
  PIPP range : 2 – 17
  PIPP mean  : 6.7
  Pain (≥7)  : 32
  No pain(<7): 47

=== MEDICATION CHECK ===
Medication sheet columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Analgesics/Anaesthetics', 'Unnamed: 4', 'Unnamed: 5', 'Broncho/Stimulants', 'Unnamed: 7']
          Unnamed: 0       Unnamed: 1         Unnamed: 2  \
0  Record identifier  Any medication?  Renal impairment?   
1             251701               No                 No   
2             251801         